# Preprocessing (converting raw images into .npz files)

## Cleaning previous output content and cloning the repo

In [1]:
import shutil, os

# Delete partially-written float32 output
output_dir = "/kaggle/working/output"
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print("Cleaned old output")

# Delete old repo clone and get fresh code with float16 changes
repo_dir = "/kaggle/working/repo"
if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)

!git clone https://github.com/TigranBoyakhchyan/GeoSpill-AI.git /kaggle/working/repo
print("Fresh repo cloned")

Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (87/87), done.
remote: Total 133 (delta 47), reused 120 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 20.41 MiB | 36.80 MiB/s, done.
Resolving deltas: 100% (47/47), done.
Fresh repo cloned


## Verifying data

In [2]:
import os, sys, subprocess
subprocess.run(["pip", "install", "-q", "rasterio"])
sys.path.insert(0, "/kaggle/working/repo/src")

IMAGES_DIR = "/kaggle/input/datasets/tigranboyakhchyan05/sar-oil-spill-raw-data/images"
MASKS_DIR  = "/kaggle/input/datasets/tigranboyakhchyan05/sar-oil-spill-raw-data/masks"

n_img = len([f for f in os.listdir(IMAGES_DIR) if f.endswith((".tif", ".tiff"))])
n_msk = len([f for f in os.listdir(MASKS_DIR) if f.endswith((".tif", ".tiff"))])
print(f"Found: {n_img} images, {n_msk} masks")

Found: 1200 images, 1200 masks


## Running the preprocessing

In [3]:
import preprocess

stats = preprocess.run(
    images_dir = IMAGES_DIR,
    masks_dir  = MASKS_DIR,
    output_dir = "/kaggle/working/output/npz_cache",
    stats_file = "/kaggle/working/output/train_stats.json",
)

SAR Oil Spill — Preprocessing

Images dir : /kaggle/input/datasets/tigranboyakhchyan05/sar-oil-spill-raw-data/images
Masks dir  : /kaggle/input/datasets/tigranboyakhchyan05/sar-oil-spill-raw-data/masks
Output dir : /kaggle/working/output/npz_cache
Stats file : /kaggle/working/output/train_stats.json

Scanning dataset...
Found 1200 image-mask pairs
Split      : 840 train | 180 val | 180 test

Computing mean/std from 840 training images...
  [ 840/840] 00591.tif
  Band 0 (VV): mean=-33.2593, std=6.1114
  Band 1 (VH): mean=-19.8870, std=4.3216
  Stats + splits saved to /kaggle/working/output/train_stats.json
  RAM after stats: 210 MB

Converting 1200 pairs in range [0, 1200) out of 1200 total...
  [  50/1200] 00297  (RAM: 196 MB)
  [ 100/1200] 00155  (RAM: 184 MB)
  [ 150/1200] 00560  (RAM: 187 MB)
  [ 200/1200] 00339  (RAM: 195 MB)
  [ 250/1200] 00177  (RAM: 194 MB)
  [ 300/1200] 01273  (RAM: 184 MB)
  [ 350/1200] 00472  (RAM: 194 MB)
  [ 400/1200] 00613  (RAM: 193 MB)
  [ 450/1200] 0007

## Verifying it all fits

In [4]:
import os

img_dir = "/kaggle/working/output/npz_cache/images"
msk_dir = "/kaggle/working/output/npz_cache/masks"

n_img = len([f for f in os.listdir(img_dir) if f.endswith(".npz")])
n_msk = len([f for f in os.listdir(msk_dir) if f.endswith(".npz")])

total_bytes = sum(os.path.getsize(os.path.join(img_dir, f)) for f in os.listdir(img_dir))
total_bytes += sum(os.path.getsize(os.path.join(msk_dir, f)) for f in os.listdir(msk_dir))

print(f"Output: {n_img} images, {n_msk} masks")
print(f"Total size: {total_bytes / 1e9:.2f} GB")
print(f"All 1200 converted: {'YES' if n_img == 1200 and n_msk == 1200 else 'NO — check for errors above'}")

# Check disk usage
!df -h /kaggle/working

Output: 1200 images, 1200 masks
Total size: 17.52 GB
All 1200 converted: YES
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   17G  3.2G  84% /kaggle/working
